# VoxMind Backend — Colab Runner

Run the cells below **in order, top to bottom**, on a GPU runtime
(Runtime -> Change runtime type -> GPU, e.g. T4).

1. Clone the repo and install Python dependencies.
2. Install and start Ollama, then pull the LLM (`llama3.2:3b`).
3. Download the Piper TTS voice files.
4. Install `cloudflared` (public tunnel client).
5. Start the VoxMind backend (FastAPI + WebSocket) in the background.
6. Open a Cloudflare Tunnel to the backend and print its public URL.

**After Cell 6 prints a `https://<something>.trycloudflare.com` URL**, take
that host and use it in the frontend's connection field as:

```
wss://<that-host>/ws
```

Note: the Colab session is ephemeral. Every time you restart the runtime
or re-run Cell 6, you get a **new** tunnel URL and must re-paste it into
the frontend.

In [ ]:
# Cell 1 — clone repo & install
!git clone <YOUR_REPO_URL> voxmind && cd voxmind && pip install -q -r backend/requirements.txt

In [ ]:
# Cell 2 — install & start Ollama, pull model
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull llama3.2:3b

In [ ]:
# Cell 3 — download a Piper voice
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx.json

In [ ]:
# Cell 4 — install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

In [ ]:
# Cell 5 — start backend in background
import os, subprocess
os.environ["PIPER_MODEL"] = "en_US-lessac-medium.onnx"
subprocess.Popen(["python", "-m", "colab.run_backend"], cwd="voxmind")

In [ ]:
# Cell 6 — open tunnel and print URL (use wss://<printed-host>/ws in the frontend)
!./cloudflared tunnel --url http://localhost:8000